In [2]:
import pathlib, sys, inspect
import numpy as np, pandas as pd

# 이 노트북은 lab/ 폴더 안에서 연다고 가정한다 (exp1_lib.py 가 있는 폴더).
LAB = pathlib.Path.cwd()
if not (LAB / "exp1_lib.py").exists():
    raise SystemExit(f"lab/ 에서 여세요. 현재 폴더에 exp1_lib.py 없음: {LAB}")

sys.path.insert(0, str(LAB))
import exp1_lib as E

# 테이블 parquet 위치. exp1_lib 기본 경로가 없으면 repo/data/tables 로 대체.
TABLE_DIR = E.DATA
if not TABLE_DIR.exists():
    TABLE_DIR = E.REPO / "data" / "tables"
E.DATA = TABLE_DIR

print("LAB       :", LAB)
print("TABLE_DIR :", TABLE_DIR, "| 존재:", TABLE_DIR.exists())
print("HARNESS   :", E.HARNESS_VERSION)
print("run_pilot :", str(inspect.signature(E.run_pilot))[:110])

LAB       : c:\Users\seude\Desktop\dp4agent\lab
TABLE_DIR : C:\Users\seude\Desktop\dp4agent\data\tables | 존재: True
HARNESS   : prefill_emul_v2
run_pilot : (client, questions: 'pd.DataFrame', ref: 'pd.DataFrame', out_csv: 'pathlib.Path', condition: 'str', n_rep: 'in


In [7]:
qs  = pd.read_csv(LAB / "questions_245.csv", encoding="utf-8-sig")
ref = pd.read_csv(LAB / "answers_ref_245.csv", encoding="utf-8-sig")

QTEXT = "How many respondents think it should be more difficult for companies to lay off workers?"
one = qs[(qs["dataset"] == "057_Spain") & (qs["question"] == QTEXT)].reset_index(drop=True)
assert len(one) == 1, one


In [8]:
cand = ref[ref["key"].astype(str).str.contains("057_Spain", regex=False)
           & ref["answer_ref"].apply(lambda v: E.close(v, 10344))]
print(cand)                                # 후보 행 확인
if len(cand) > 1:                          # 여러 개면 147/146 번호로 좁힘
    cand = cand[cand["key"].astype(str).str.contains(r"\|14[67]$", regex=True)]
assert len(cand) == 1, cand

one = one.assign(n=int(cand["n"].iloc[0]))
print(one[["n", "dataset", "row", "answer"]])


       n                key  answer_gold  answer_ref match
212  213  057_Spain|dev|146      10344.0     10344.0     O
     n    dataset  row   answer
0  213  057_Spain  146  10344.0


In [9]:
client = E.make_client()
COND = "REP10_enone_T0"
runs = E.run_pilot(
    client, one, ref,
    out_csv="rep10_057_Spain_147.csv",     # lab/result/ 아래에 저장
    condition=COND, n_rep=10,
    model=E.MODEL_DEFAULT, temperature=0,
    max_tokens=300, reasoning_effort="none",
)


  REP10_enone_T0 n=213 rep=1 O exec=ok fr=stop     out=99 rea=0
  REP10_enone_T0 n=213 rep=2 X exec=ok fr=stop     out=74 rea=0
  REP10_enone_T0 n=213 rep=3 X exec=ok fr=stop     out=78 rea=0
  REP10_enone_T0 n=213 rep=4 O exec=ok fr=stop     out=98 rea=0
  REP10_enone_T0 n=213 rep=5 O exec=ok fr=stop     out=98 rea=0
  REP10_enone_T0 n=213 rep=6 X exec=ok fr=stop     out=74 rea=0
  REP10_enone_T0 n=213 rep=7 O exec=ok fr=stop     out=98 rea=0
  REP10_enone_T0 n=213 rep=8 O exec=ok fr=stop     out=78 rea=0
  REP10_enone_T0 n=213 rep=9 O exec=ok fr=stop     out=98 rea=0
  REP10_enone_T0 n=213 rep=10 X exec=ok fr=stop     out=74 rea=0


In [10]:
r = runs[runs["condition"] == COND]
print(pd.Series(E.summarize(r)))
print("\n응답값 분포:\n", r["answer_agent"].astype(str).value_counts())
print("\n해석(서명) 분포:\n", r["sig_key"].value_counts())

for _, x in r[~r["correct"].astype(bool)].iterrows():
    print(f"\n--- rep {x.rep} | 값={x.answer_agent} | 오류={x.exec_err} ---\n{x.body}")


호출 수                10.0000
호출 실패                0.0000
finish=length        0.0000
본문 비어있음              0.0000
실행 성공률 %           100.0000
실행 오류 건수             0.0000
값 None %             0.0000
하네스 위반 %             0.0000
서명 추출 %            100.0000
정답률 %               60.0000
값 일관성 %             60.0000
의도 일관성 %            60.0000
응답시간 평균s             2.2000
응답시간 합계min           0.4000
출력토큰 평균             86.9000
사고토큰 평균              0.0000
캐시hit 평균         20000.7000
fingerprint 수        0.0000
비용 USD               0.0047
dtype: float64

응답값 분포:
 answer_agent
10344    6
19910    4
Name: count, dtype: int64

해석(서명) 분포:
 sig_key
[["It should be more difficult for companies to lay off workers"], [], ["sum"]]      6
[["It should be more difficult for companies to lay off workers"], [], ["count"]]    4
Name: count, dtype: int64

--- rep 2 | 값=19910 | 오류=nan ---
    # The columns used to answer the question: ['It should be more difficult for companies to lay off workers']
    # The types